# Food Calorie Prediction from Images

This notebook implements a Convolutional Neural Network (CNN) model to predict the calorie content of food items from images. We'll use the NutritionVerse-Real dataset, which contains 889 images of 251 distinct dishes across 45 unique food types.

## Project Structure
1. Dataset Download & Exploration
2. Data Preprocessing & Augmentation 
3. CNN Model Architecture Design
4. Model Training & Evaluation
5. Gradio Interface Integration

Let's begin!

## 1. Environment Setup & Dataset Download

First, let's install the necessary packages and download the dataset from Kaggle.

In [ ]:
# Install required packages
!pip install tensorflow numpy pandas matplotlib seaborn scikit-learn kaggle gradio pillow

# Set up Kaggle API credentials (you need to upload your kaggle.json file)
import os
import json

# Uncomment the following if you're in Google Colab
# uploaded = files.upload()  # Upload kaggle.json
# os.makedirs('/root/.kaggle', exist_ok=True)
# with open('/root/.kaggle/kaggle.json', 'w') as f:
#     f.write(json.dumps(json.loads(next(iter(uploaded.values())).decode())))
# os.chmod('/root/.kaggle/kaggle.json', 600)

# For local environment, ensure your kaggle.json is in ~/.kaggle/ with proper permissions


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached semantic_version-2.10.0-py2.py3-none-any.whl.metadata (9.7 kB)
   ---------------------------------------- 0.0/54.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/54.1 MB ? eta -:--:--
   - -------------------------------------- 1.6/54.1 MB 5.2 MB/s eta 0:00:11
   -- ------------------------------------- 3.9/54.1 MB 8.4 MB/s eta 0:00:06
   -------- ------------------------------- 11.0/54.1 MB 15.6 MB/s eta 0:00:03
   -------- ------------------------------- 12.1/54.1 MB 16.4 MB/s eta 0:00:03
   ---------------- ----------------------- 22.8/54.1 MB 20.6 MB/s eta 0:00:02
   ------------------ --------------------- 24.9/54.1 MB 19.2 MB/s eta 0:00:02
   --------------------------- ------------ 37.5/54.1 MB 24.8 MB/s eta 0:00:01
   ---------------------------------- ----- 46.4/54.1 MB 27.1 MB/s eta 0:00:01
   ---------------------------------------  54.0/54.1 MB 28.4 MB/s eta 0:00:01
   ---------------------------------------- 54.1/54.1 MB 27.4 MB/s eta 0:00

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# Download the NutritionVerse-Real dataset
!kaggle datasets download -d nutritionverse/nutritionverse-real --force
!unzip -q nutritionverse-real.zip -d nutritionverse-data

## 2. Data Exploration & Understanding

Let's explore the dataset structure and examine some sample images and metadata.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Setting paths
base_path = 'nutritionverse-data'
images_path = os.path.join(base_path, 'images')
metadata_path = os.path.join(base_path, 'metadata.csv')

# Load metadata
metadata_df = pd.read_csv(metadata_path)

# Display first few rows of metadata
print("Dataset Metadata Overview:")
print(f"Total entries: {len(metadata_df)}")
metadata_df.head()

In [ ]:
# Exploring the distribution of food types and nutritional values
plt.figure(figsize=(12, 5))

# Distribution of calories
plt.subplot(1, 2, 1)
sns.histplot(metadata_df['calories'], kde=True)
plt.title('Distribution of Calories')
plt.xlabel('Calories')

# Top 10 food types by count
plt.subplot(1, 2, 2)
food_type_counts = metadata_df['food_type'].value_counts().head(10)
sns.barplot(y=food_type_counts.index, x=food_type_counts.values)
plt.title('Top 10 Food Types')
plt.xlabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Function to display sample images with their calorie information
def display_sample_images(num_samples=5):
    sample_indices = np.random.choice(len(metadata_df), num_samples, replace=False)
    
    plt.figure(figsize=(15, 10))
    for i, idx in enumerate(sample_indices):
        row = metadata_df.iloc[idx]
        img_path = os.path.join(images_path, row['image_name'])
        
        if os.path.exists(img_path):
            img = Image.open(img_path)
            plt.subplot(1, num_samples, i+1)
            plt.imshow(img)
            plt.title(f"Food: {row['food_type']}\nCalories: {row['calories']}")
            plt.axis('off')
    
    plt.tight_layout()
    plt.show()

# Display sample images
display_sample_images()

## 3. Data Preprocessing

Now, let's preprocess the images and metadata for our model.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Constants
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32

# Function to load and preprocess images
def load_image(img_path):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_image(img, channels=3)
    img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])
    img = img / 255.0  # Normalize to [0,1]
    return img

# Create a path column for images
metadata_df['image_path'] = metadata_df['image_name'].apply(lambda x: os.path.join(images_path, x))

# Check for missing images and filter out rows with missing images
valid_images = []
for img_path in metadata_df['image_path']:
    if os.path.exists(img_path):
        valid_images.append(True)
    else:
        valid_images.append(False)

metadata_df['valid_image'] = valid_images
filtered_df = metadata_df[metadata_df['valid_image']].copy()
print(f"Total valid images: {filtered_df.shape[0]} out of {metadata_df.shape[0]}")

# Split data into training, validation, and test sets
train_df, test_val_df = train_test_split(filtered_df, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(test_val_df, test_size=0.5, random_state=42)

print(f"Training set: {train_df.shape[0]} images")
print(f"Validation set: {val_df.shape[0]} images")
print(f"Test set: {test_df.shape[0]} images")

# Scale calorie values
scaler = StandardScaler()
train_df['scaled_calories'] = scaler.fit_transform(train_df[['calories']])
val_df['scaled_calories'] = scaler.transform(val_df[['calories']])
test_df['scaled_calories'] = scaler.transform(test_df[['calories']])

In [ ]:
# Create data generators with augmentation for training
train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2
)

val_datagen = ImageDataGenerator()
test_datagen = ImageDataGenerator()

# Function to create dataset from dataframe
def create_dataset(df, datagen, batch_size=BATCH_SIZE, is_train=False):
    images = []
    calories = []
    
    for i, row in df.iterrows():
        try:
            img_path = row['image_path']
            img = load_image(img_path).numpy()
            images.append(img)
            calories.append(row['scaled_calories'])
            
            # Apply augmentation for training
            if is_train:
                for _ in range(2):  # Generate 2 augmented images for each original
                    aug_img = datagen.random_transform(img)
                    images.append(aug_img)
                    calories.append(row['scaled_calories'])
        except Exception as e:
            print(f"Error processing {row['image_path']}: {e}")
    
    return np.array(images), np.array(calories)

# Create datasets
train_images, train_calories = create_dataset(train_df, train_datagen, is_train=True)
val_images, val_calories = create_dataset(val_df, val_datagen)
test_images, test_calories = create_dataset(test_df, test_datagen)

print(f"Training images shape: {train_images.shape}")
print(f"Validation images shape: {val_images.shape}")
print(f"Test images shape: {test_images.shape}")

## 4. CNN Model Building

Let's build a CNN model for calorie prediction. We'll use transfer learning with a pre-trained model as the base.

In [ ]:
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

def build_model():
    # Create a vanilla CNN model
    model = Sequential([
        # First convolutional block
        Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        # Second convolutional block
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        # Third convolutional block
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        # Fully connected layers
        Flatten(),
        Dense(512, activation='relu'),
        Dropout(0.5),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(1)  # Regression output (calorie prediction)
    ])
    
    # Compile model
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    
    return model

# Build the model
calorie_model = build_model()
calorie_model.summary()

## 5. Model Training & Evaluation

Let's train the model and evaluate its performance.

In [ ]:
# Set up callbacks
checkpoint_path = "best_model.h5"
callbacks = [
    EarlyStopping(patience=10, restore_best_weights=True, verbose=1),
    ModelCheckpoint(checkpoint_path, save_best_only=True, verbose=1)
]

# Train the model
history = calorie_model.fit(
    train_images, train_calories,
    validation_data=(val_images, val_calories),
    epochs=50,
    batch_size=BATCH_SIZE,
    callbacks=callbacks
)

In [ ]:
# Plot training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['mae'], label='Training MAE')
plt.plot(history.history['val_mae'], label='Validation MAE')
plt.title('Mean Absolute Error over Epochs')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
test_loss, test_mae = calorie_model.evaluate(test_images, test_calories)
print(f"Test Loss (MSE): {test_loss:.4f}")
print(f"Test MAE: {test_mae:.4f}")

# Make predictions on test set
test_preds = calorie_model.predict(test_images)

# Convert scaled predictions back to original scale
test_preds_original = scaler.inverse_transform(test_preds)
test_calories_original = scaler.inverse_transform(test_calories.reshape(-1, 1))

# Calculate metrics in original scale
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(test_calories_original, test_preds_original)
mse = mean_squared_error(test_calories_original, test_preds_original)
rmse = np.sqrt(mse)
r2 = r2_score(test_calories_original, test_preds_original)

print(f"MAE (original scale): {mae:.2f} calories")
print(f"RMSE (original scale): {rmse:.2f} calories")
print(f"R² Score: {r2:.4f}")

In [ ]:
# Visualize actual vs predicted calorie values
plt.figure(figsize=(10, 6))
plt.scatter(test_calories_original, test_preds_original, alpha=0.7)
plt.plot([min(test_calories_original), max(test_calories_original)], 
         [min(test_calories_original), max(test_calories_original)], 
         'r--', lw=2)
plt.xlabel('Actual Calories')
plt.ylabel('Predicted Calories')
plt.title('Actual vs Predicted Calorie Values')
plt.grid(True)
plt.show()

# Display sample predictions
def display_sample_predictions(num_samples=5):
    sample_indices = np.random.choice(len(test_images), num_samples, replace=False)
    
    plt.figure(figsize=(15, 10))
    for i, idx in enumerate(sample_indices):
        img = test_images[idx]
        actual = test_calories_original[idx][0]
        predicted = test_preds_original[idx][0]
        
        plt.subplot(1, num_samples, i+1)
        plt.imshow(img)
        plt.title(f"Actual: {actual:.0f} cal\nPredicted: {predicted:.0f} cal")
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()

display_sample_predictions()

## 6. Fine-tuning and Improving the Model

Let's fine-tune our model by unfreezing some layers of the base model and training with a lower learning rate.

In [ ]:
# For fine-tuning our vanilla CNN, we'll create a deeper version with lower learning rate
def build_fine_tuned_model():
    # Create a deeper vanilla CNN model
    model = Sequential([
        # First convolutional block
        Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        # Second convolutional block
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        # Third convolutional block
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        # Fourth convolutional block (additional)
        Conv2D(256, (3, 3), activation='relu', padding='same'),
        Conv2D(256, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        # Fully connected layers
        Flatten(),
        Dense(512, activation='relu'),
        Dropout(0.5),
        Dense(256, activation='relu'),
        Dropout(0.4),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(1)  # Regression output (calorie prediction)
    ])
    
    # Compile model with a lower learning rate
    model.compile(
        optimizer=Adam(learning_rate=0.0001),  # Lower learning rate for fine-tuning
        loss='mse',
        metrics=['mae']
    )
    
    return model

# Build the fine-tuned model
fine_tuned_model = build_fine_tuned_model()

# Train with fine-tuning
fine_tuning_history = fine_tuned_model.fit(
    train_images, train_calories,
    validation_data=(val_images, val_calories),
    epochs=20,
    batch_size=BATCH_SIZE,
    callbacks=callbacks
)

In [ ]:
# Evaluate the fine-tuned model
ft_test_loss, ft_test_mae = fine_tuned_model.evaluate(test_images, test_calories)
print(f"Fine-tuned Test Loss (MSE): {ft_test_loss:.4f}")
print(f"Fine-tuned Test MAE: {ft_test_mae:.4f}")

# Make predictions with fine-tuned model
ft_test_preds = fine_tuned_model.predict(test_images)
ft_test_preds_original = scaler.inverse_transform(ft_test_preds)

# Calculate metrics in original scale for fine-tuned model
ft_mae = mean_absolute_error(test_calories_original, ft_test_preds_original)
ft_mse = mean_squared_error(test_calories_original, ft_test_preds_original)
ft_rmse = np.sqrt(ft_mse)
ft_r2 = r2_score(test_calories_original, ft_test_preds_original)

print(f"Fine-tuned MAE (original scale): {ft_mae:.2f} calories")
print(f"Fine-tuned RMSE (original scale): {ft_rmse:.2f} calories")
print(f"Fine-tuned R² Score: {ft_r2:.4f}")

# Compare initial model vs fine-tuned model
print("\nImprovement after fine-tuning:")
print(f"MAE improvement: {mae - ft_mae:.2f} calories")
print(f"RMSE improvement: {rmse - ft_rmse:.2f} calories")
print(f"R² improvement: {ft_r2 - r2:.4f}")

## 7. Save the Model for Deployment

Let's save the trained model for later use in our Gradio interface.

In [ ]:
# Save the best model
best_model = fine_tuned_model  # Or use the initial model if it performed better
best_model.save('food_calorie_model.h5')

# Save the scaler for preprocessing new data
import joblib
joblib.dump(scaler, 'calorie_scaler.pkl')

print("Model and scaler saved successfully!")

## 8. Interactive Interface with Gradio

Finally, let's create an interactive interface for our model using Gradio.

In [ ]:
import gradio as gr
import tensorflow as tf
from PIL import Image as PILImage
import numpy as np
import joblib

# Load the saved model and scaler
loaded_model = tf.keras.models.load_model('food_calorie_model.h5')
loaded_scaler = joblib.load('calorie_scaler.pkl')

# Function to preprocess the image
def preprocess_image(image):
    img = PILImage.fromarray(image).resize((IMG_HEIGHT, IMG_WIDTH))
    img_array = np.array(img) / 255.0
    return np.expand_dims(img_array, axis=0)

# Function to predict calories
def predict_calories(image):
    if image is None:
        return "No image provided"
    
    # Preprocess the image
    preprocessed_img = preprocess_image(image)
    
    # Make prediction
    scaled_prediction = loaded_model.predict(preprocessed_img)[0][0]
    
    # Convert prediction back to original scale
    prediction = loaded_scaler.inverse_transform([[scaled_prediction]])[0][0]
    
    return f"Estimated calories: {prediction:.0f} kcal"

# Create Gradio interface
demo = gr.Interface(
    fn=predict_calories,
    inputs=gr.Image(),
    outputs=gr.Textbox(),
    title="Food Calorie Predictor",
    description="Upload an image of food to predict its calorie content.",
    examples=[
        # Add paths to example images if you have them
    ],
    theme="huggingface"
)

# Launch the interface
demo.launch(share=True)

## 9. Conclusion & Future Improvements

We've successfully built a CNN model that predicts the calorie content of food items from images. The model was trained on the NutritionVerse-Real dataset and deployed as an interactive web application using Gradio.

### Key Accomplishments
- Preprocessed and augmented the dataset for effective training
- Implemented a transfer learning approach using MobileNetV2
- Fine-tuned the model to improve performance
- Created an interactive user interface for real-time predictions

### Future Improvements
1. **Data Expansion**: Include more diverse food images to improve generalization
2. **Model Enhancements**: Experiment with different architectures and hyperparameters
3. **Multi-output Prediction**: Extend the model to predict other nutritional values (protein, carbs, fat)
4. **Mobile Deployment**: Convert the model to TensorFlow Lite for mobile applications
5. **Context Awareness**: Incorporate portion size estimation for more accurate predictions

### References
- NutritionVerse-Real dataset: https://www.kaggle.com/datasets/nutritionverse/nutritionverse-real/data
- TensorFlow: https://www.tensorflow.org/
- Gradio: https://www.gradio.app/